## Learning notes — 12. Transform Refunds Data

### Learning objectives
Follow **12. Transform Refunds Data** from its source inputs through its operations to the archived results.
- Explain schema, ctas, profiling as used in this lesson.

### Source-specific learning focus
- Compare raw fields with typed records and parsing failures.
- Distinguish query results from materialized table state.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `hive_metastore.bronze.refunds`, `hive_metastore.silver.refunds`.


## Learning notes — corrective commentary (originals unchanged)

Refunds setup SQL contains a NOT NULL. typo: the period must be removed in a separate corrected copy before database setup. The supplied SQL is setup, not remote Refunds table contents.

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
1. Extract specific portion of the string from refund_reason using regexp_extract function
1. Extract date and time from the refund_timestamp
1. Write transformed data to the Silver schema in hive metastore [Default Location: /user/hive/warehouse]

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT refund_id,
       payment_id,
       refund_timestamp, 
       refund_amount,
       refund_reason
  FROM hive_metastore.bronze.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
refund_id	payment_id	refund_timestamp	refund_amount	refund_reason
1	66	2025-01-10T11:30:00Z	85.75	Payment Error:Retailer
2	69	2025-01-03T12:40:15Z	120.50	Order Cancelled:Customer
3	72	2025-01-06T14:45:30Z	65.00	Product Returned:Customer
4	73	2025-01-07T16:10:45Z	210.99	Order Cancelled:Customer
5	75	2025-01-09T18:25:00Z	45.20	Payment Error:Retailer
6	80	2025-01-10T09:35:20Z	130.15	Order Cancelled:Customer
7	83	2025-01-12T11:20:40Z	150.00	Product Returned:Customer
8	85	2025-01-14T13:15:30Z	89.99	Order Cancelled:Customer
9	89	2025-01-15T15:00:00Z	78.50	Payment Error:Retailer
10	91	2025-01-17T16:45:15Z	250.75	Product Returned:Customer

### 1. Extract specific portion of the string from refund_reason using split function
> [Documentation for split Function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/split)

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT refund_id,
       payment_id,
       refund_timestamp, 
       refund_amount,
       SPLIT(refund_reason, ':')[0] AS refund_reason,
       SPLIT(refund_reason, ':')[1] AS refund_
  FROM hive_metastore.bronze.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
refund_id	payment_id	refund_timestamp	refund_amount	refund_reason	refund_source
1	66	2025-01-10T11:30:00Z	85.75	Payment Error	Retailer
2	69	2025-01-03T12:40:15Z	120.50	Order Cancelled	Customer
3	72	2025-01-06T14:45:30Z	65.00	Product Returned	Customer
4	73	2025-01-07T16:10:45Z	210.99	Order Cancelled	Customer
5	75	2025-01-09T18:25:00Z	45.20	Payment Error	Retailer
6	80	2025-01-10T09:35:20Z	130.15	Order Cancelled	Customer
7	83	2025-01-12T11:20:40Z	150.00	Product Returned	Customer
8	85	2025-01-14T13:15:30Z	89.99	Order Cancelled	Customer
9	89	2025-01-15T15:00:00Z	78.50	Payment Error	Retailer
10	91	2025-01-17T16:45:15Z	250.75	Product Returned	Customer

### 2. Extract specific portion of the string from refund_reason using regexp_extract function
> [Documentation for regexp_extract Function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_extract)  
> [Regex Pattern](https://docs.oracle.com/javase/7/docs/api/java/util/regex/Pattern.html)

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT refund_id,
       payment_id,
       CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
       date_format(refund_timestamp, 'HH:mm:ss') AS refund_time,
       refund_amount,
       regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
       regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
  FROM hive_metastore.bronze.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
refund_id	payment_id	refund_date	refund_time	refund_amount	refund_reason	refund_source
1	66	2025-01-10	11:30:00	85.75	Payment Error	Retailer
2	69	2025-01-03	12:40:15	120.50	Order Cancelled	Customer
3	72	2025-01-06	14:45:30	65.00	Product Returned	Customer
4	73	2025-01-07	16:10:45	210.99	Order Cancelled	Customer
5	75	2025-01-09	18:25:00	45.20	Payment Error	Retailer
6	80	2025-01-10	09:35:20	130.15	Order Cancelled	Customer
7	83	2025-01-12	11:20:40	150.00	Product Returned	Customer
8	85	2025-01-14	13:15:30	89.99	Order Cancelled	Customer
9	89	2025-01-15	15:00:00	78.50	Payment Error	Retailer
10	91	2025-01-17	16:45:15	250.75	Product Returned	Customer

### 3. Write transformed data to the Silver schema  

## Learning notes — before source operation

**schema**: Compare declared or inferred field types with raw input. Parsing, schema evolution and rescued/corrupt records can produce nulls or additional columns rather than clean typed rows.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Parsing a numeric field from an incompatible string can fail or yield null depending on parsing mode; inspect the declared schema and error policy.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE SCHEMA hive_metastore.silver;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

## Learning notes — before source operation

**ctas**: Table creation materializes a query/DataFrame into a destination. Distinguish its stored schema and rows from a view and check whether replacement overwrites existing state.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE hive_metastore.silver.refunds
AS
SELECT refund_id,
       payment_id,
       CAST(date_format(refund_timestamp, 'yyyy-MM-dd') AS DATE) AS refund_date,
       date_format(refund_timestamp, 'HH:mm:ss') AS refund_time,
       refund_amount,
       regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
       regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
  FROM hive_metastore.bronze.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):
num_affected_rows	num_inserted_rows

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM hive_metastore.silver.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
refund_id	payment_id	refund_date	refund_time	refund_amount	refund_reason	refund_source
1	66	2025-01-10	11:30:00	85.75	Payment Error	Retailer
2	69	2025-01-03	12:40:15	120.50	Order Cancelled	Customer
3	72	2025-01-06	14:45:30	65.00	Product Returned	Customer
4	73	2025-01-07	16:10:45	210.99	Order Cancelled	Customer
5	75	2025-01-09	18:25:00	45.20	Payment Error	Retailer
6	80	2025-01-10	09:35:20	130.15	Order Cancelled	Customer
7	83	2025-01-12	11:20:40	150.00	Product Returned	Customer
8	85	2025-01-14	13:15:30	89.99	Order Cancelled	Customer
9	89	2025-01-15	15:00:00	78.50	Payment Error	Retailer
10	91	2025-01-17	16:45:15	250.75	Product Returned	Customer

Archived Databricks source — shown for reading, not Python execution.

```sql
DESC EXTENDED hive_metastore.silver.refunds;
```

In [ ]:
# Archived output only; no computation.

Archived table:
col_name	data_type	comment
refund_id	int	None
payment_id	int	None
refund_date	date	None
refund_time	string	None
refund_amount	decimal(10,2)	None
refund_reason	string	None
refund_source	string	None
		
# Delta Statistics Columns		
Column Names	refund_id, refund_source, refund_amount, refund_reason, payment_id, refund_time, refund_date	
Column Selection Method	first-32	
		
# Detailed Table Information		
Catalog	hive_metastore	
Database	silver	
Table	refunds	
Created Time	Mon Dec 23 14:51:15 UTC 2024	
Last Access	UNKNOWN	
Created By	Spark 3.5.0	
Type	MANAGED	
Location	dbfs:/user/hive/warehouse/silver.db/refunds	
Provider	delta	
Owner	root	
Is_managed_location	true	
Table Properties	[delta.enableDeletionVectors=true,delta.feature.deletionVectors=supported,delta.minReaderVersion=3,delta.minWriterVersion=7]	